# Merging with Different Column Names

<a href="https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week09/04.Merging-with-Different-Column-Names/notebooks/01_04.Merging-with-Different-Column-Names.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview
In enterprise IT environments, different departments and software systems rarely use identical column names for the same entity.
- The Student Administration system may identify students by `student_id`.
- The Finance Billing system may call the exact same identifier `id_number` or `account_no`.

In this notebook, you will learn how to:
- Join tables with mismatched key names using `left_on` and `right_on`.
- Clean up redundant duplicate key columns after merging.
- Join a table's column against another table's index (`right_index=True`).
- Disambiguate overlapping non-key column names using custom `suffixes`.

## 1. Setup: Creating Disparate Departmental Tables
Construct Academic Portal and Finance Billing tables.

In [ ]:
import pandas as pd

portal_df = pd.DataFrame({
    'student_id': [101, 102, 103, 104],
    'full_name': ['Liam Nguyen', 'Emma Watson', 'Oliver Brown', 'Sophia Vu'],
    'major': ['Computer Science', 'Information Technology', 'Data Science', 'Software Engineering'],
    'status': ['Active', 'Active', 'Probation', 'Active']
})

finance_df = pd.DataFrame({
    'id_number': [101, 102, 103, 105],
    'tuition_due_aud': [4500.0, 0.0, 3200.0, 1800.0],
    'payment_plan': ['Upfront', 'HECS-HELP', 'Installments', 'Upfront'],
    'status': ['Current', 'Current', 'Overdue', 'Current']
})

display(portal_df)
display(finance_df)

## 2. Joining on Mismatched Column Names (left_on & right_on)
Specify the key columns on both sides using `left_on` and `right_on`.

In [ ]:
merged_raw = pd.merge(
    portal_df,
    finance_df,
    left_on='student_id',
    right_on='id_number',
    how='left'
)
display(merged_raw)

## 3. Cleaning Up Redundant Key Columns
Notice above that both `student_id` and `id_number` exist in the result. Keep schemas clean by dropping the secondary key.

In [ ]:
merged_clean = merged_raw.drop(columns=['id_number'])
display(merged_clean)

## 4. Disambiguating Overlapping Columns with suffixes
Both tables contain a column named `status` (Academic Status vs Billing Status). Assign self-documenting suffixes.

In [ ]:
merged_suffixed = pd.merge(
    portal_df,
    finance_df,
    left_on='student_id',
    right_on='id_number',
    how='left',
    suffixes=('_academic', '_billing')
).drop(columns=['id_number'])

display(merged_suffixed[['student_id', 'full_name', 'status_academic', 'status_billing', 'tuition_due_aud']])

## 5. Merging with an Index (right_index=True)
Join directly against an indexed table.

In [ ]:
finance_indexed = finance_df.set_index('id_number')
index_merged = pd.merge(
    portal_df,
    finance_indexed,
    left_on='student_id',
    right_index=True,
    how='left',
    suffixes=('_portal', '_finance')
)
display(index_merged[['student_id', 'full_name', 'tuition_due_aud', 'payment_plan']])

## Enrichment
### Merging on Multiple Mismatched Keys
```python
# pd.merge(df1, df2, left_on=['campus_id', 'term'], right_on=['campus_code', 'semester'])
```

## Takeaways
- Use `left_on='key1'` and `right_on='key2'` when joining columns with differing header names.
- Always drop the redundant secondary key column after merging to avoid confusing users.
- Use `suffixes=('_label1', '_label2')` to cleanly distinguish overlapping column names.
- Use `left_index=True` or `right_index=True` when joining against a DataFrame's index.

## Conclusion
Real-world data integration requires mapping between varying column schemas. Using `left_on`, `right_on`, and explicit `suffixes` ensures consistent, readable analytical views.

## Exercises
**Exercise 1:** Merge `portal_df` and `finance_df` using `how='inner'`, `left_on='student_id'`, and `right_on='id_number'`.

**Exercise 2:** Apply custom suffixes `('_portal', '_finance')` to the inner merge and drop `'id_number'`.

**Exercise 3:** Filter the merged DataFrame for students who have an overdue billing status (`status_finance == 'Overdue'`).

In [ ]:
# Write your practice code here

# --- Solutions ---
# inner_clean = pd.merge(
#     portal_df,
#     finance_df,
#     left_on='student_id',
#     right_on='id_number',
#     how='inner',
#     suffixes=('_portal', '_finance')
# ).drop(columns=['id_number'])
# display(inner_clean)
#
# overdue = inner_clean[inner_clean['status_finance'] == 'Overdue']
# display(overdue)